# Quantitative ALM Engine: Behavioral EVE Stress Testing
**Interest Rate Risk in the Banking Book (IRRBB)**

This notebook demonstrates a production-grade Asset-Liability Management (ALM) pipeline. Traditional ALM models rely on static cash flows and flat deposit betas, which fail during extreme rate shocks.

This engine bridges the traditional quantitative finance with modern Machine Learning by implementing:
1. **Assets:** A Hybrid Mortgage Prepayment Model (Structural S-Curve Anchor + ML Residuals).
2. **Liabilities:** An Asymetric Non-Maturing Deposit (NMD) Beta Model with Zero-Lower Bound enforcement.

The final output is a dynamic Economic Value of Equity (EVE) stress test that captures the negative convexity of callable mortgages and the margin squeeze of deposit flight.

In [ ]:
import sys
import os
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, "..")) if "src" not in os.listdir(current_dir) else current_dir

if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.behavioral_models.base_cpr_calculator import BaseCPRCalculator
from src.behavioral_models.prepayment_calibrator import HybridPrepaymentCalibrator
from src.behavioral_models.nmd_calibrator import NMDBetaCalibrator
from src.behavioral_models.nmd_beta_model import NMDBetaModel
from src.alm_engine.yield_curve import YieldCurve
from src.alm_engine.instruments import RetailMortgage, NonMaturingDeposit
from src.alm_engine.engine import ALMEngine

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'

## 1. The Asset Side: Hybrid Mortgage Prepayments

Pure Machine Learning extrapolates unpredictably under severe regulatory shocks (e.g. +/- 400 bps), often violating Model Risk Management (MRM) standards. To guarantee structural safety, this pipeline uses a two-step calibrator:
* **The Anchor:** Regresses loan-level servicing tapes into a Sigmoid S-Curve to discover the absolute macroeconomic boundaries (Base CPR, Max CPR).
* **The ML Modifier:** Trains a Scikit-Learn 'RandomForestRegressor' strictly on the *residual errors* of the S-Curve to capture borrower-level covariates like FICO and Burnout.

In [ ]:
print("--- Initializing ETL & Mortgage Prepayment Calibration ---")
# 1. Generate synthetic loan-level servicing tape
calculator = BaseCPRCalculator(random_seed=42)
raw_tape = calculator.simulate_servicing_tape(num_records=25000)
tape_with_smm = calculator.calculate_loan_level_smm(raw_tape)

# 2. Extract empirical baseline and fit the Hybrid ML model
base_cpr = calculator.extract_base_cpr(tape_with_smm)
hybrid_model = HybridPrepaymentCalibrator(base_cpr).fit_model(tape_with_smm)

print(f"Base CPR anchored at: {base_cpr * 100:.2f}%")
print("Hybrid Prepayment Model: SUCCESSFULLY FITTED (S-Curve + Random Forest)") 

contractual_rate = 0.05
market_rates = np.linspace(0.01, 0.08, 100)
incentives = contractual_rate - market_rates

prime_cprs = [hybrid_model.calculate_cpr(contractual_rate, mkt, fico=800, burnout=0) for mkt in market_rates]
subprime_cprs = [hybrid_model.calculate_cpr(contractual_rate, mkt, fico=620, burnout=1) for mkt in market_rates]

plt.figure(figsize=(10, 6))
plt.style.use('bmh')

plt.plot(incentives * 10000, np.array(prime_cprs) * 100, label="Prime (FICO 800, No Burnout)", color='darkblue', linewidth=2.5)
plt.plot(incentives * 10000, np.array(subprime_cprs) * 100, label="Subprime (FICO 620, Burned Out)", color='darkred', linewidth=2.5, linestyle='--')

plt.axhline(hybrid_model.base_cpr * 100, color='gray', linestyle=':', label=f'MRM Lower Bound (Base CPR: {hybrid_model.base_cpr*100:.1f}%)')
plt.axhline(hybrid_model.max_cpr *100, color='black', linestyle=':', label=f'MRM Upper Bound (Max CPR: {hybrid_model.max_cpr*100:.1f}%)')

plt.title('Hybrid S-Curve Prepayment Dynamics: ML Covariate Impact', fontsize=14, fontweight='bold')
plt.xlabel('Refinancing Incentive (bps)', fontsize=12)
plt.ylabel('Conditional Prepayment Rate (CPR %)', fontsize=12)
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.show()

## 2. The Liability Side: Asymmetric NMD Betas

Retail deposits (Checking/Savings) do not have contractual maturities. Bank management treats them dynamically based on their macroeconomic cycle.
* **Up-Cycle:** When market rates rise, banks hike deposit rates slowly (e.g. 30% Beta) to maximize Net Interest Margin.
* **Down-Cycle:** When market rates fall, banks cut deposit rates aggressively (e.g. 70% Beta) to protect margins, strictly stopping at the **Zero Lower Bound (0.00%)**.

Below, I simulate 5 years of historical rate data and use Ordinary Least Squares (OLS) to successfully extract the bank's true underlying asymmetric betas.

In [ ]:
# 1. Generate 5 years (60 months) of historical rate cycles
nmd_calibrator = NMDBetaCalibrator(random_seed=100)
history_df = nmd_calibrator.generate_synthetic_history(periods=60)

# 2. Calibrate Asymmetric Betas using OLS regression
nmd_model = nmd_calibrator.calibrate_betas(history_df)

print("--- NMD Beta Calibration Results ---")
print(f"Extracted Up-Beta: {nmd_model.up_beta * 100:.1f}%")
print(f"Extracted Down-Beta: {nmd_model.down_beta * 100:.1f}%")

# 3. Visualize the Asymmetric Pass-through Behavior
plt.figure(figsize=(10, 4))
plt.plot(history_df['month'], history_df['market_rate'] * 100, label='Market Rate (e.g. 3M SOFR)', color='navy', linewidth=2)
plt.plot(history_df['month'], history_df['deposit_rate'] * 100, label='Bank Deposit Rate', color='darkorange', linewidth=2, linestyle='--')
plt.title("Historical Rate Cycles: Asymmetric Deposit Pass-Through", fontsize=12, fontweight='bold')
plt.ylabel("Interest Rate (%)")
plt.xlabel("Month")
plt.legend()
plt.tight_layout()
plt.show()

## 3. Balance Sheet Synthesis: Economic Value of Equity (EVE)

The true test of an ALM Engine is how assets and liabilities interact under Basel III / regulatory stress scenarios. I will evaluate a $200M  mortgage portfolio funded by $180M in deposits across a -300 bps to +300 bps shock range.

To perform a true risk attribution analysis, I plot a 4-bank matrix:
1. **Bank A (Static):** Assumes a naive, static deposit rate regardless of market shocks.
2. **Bank B (Dynamic Assets):** Uses the Hybrid ML Prepayment Model, but static deposits. Isolates **Negative Convexity** (borrowers refinancing when rates fall).
3. **Bank C (Dynamic Liabilities):** Uses the Asymmetric NMD Beta Model, but static prepayments. Isolates the **Margin Squeeze** (depositors demanding higher yields when rates rise).
4. **Bank D (Both Dynamic):** Combnines ML Prepayments and Asymmetric NMD Betas. Reveals the true, dual-tailed "frown" shape of IRRBB.

In [ ]:
# 1. Create a Naive Static Prepayment Model for Banks A and C
class StaticPrepaymentModel:
    def calculate_cpr(self, contractual_rate, market_rate, fico, burnout):
        return 0.10 # Flat 10% CPR regardless of market shocks

static_prepayments = StaticPrepaymentModel()

# 2. Set up the portfolios
tenors = [0.25, 1.0, 5.0, 10.0, 30.0]
base_rates = [0.04, 0.04, 0.04, 0.04, 0.04]

# Assets (Naive vs Machine Learning)
naive_mortgages = RetailMortgage(
    notional=200_000_000, rate=0.05, maturity=10.0,
    prepayment_model=static_prepayments, fico=750, burnout=0
)
ml_mortgages = RetailMortgage(
    notional=200_000_000, rate=0.05, maturity=10.0,
    prepayment_model=hybrid_model, fico=750, burnout=0
)

# Liabilities (Static vs Dynamic Betas)
static_deposits = NonMaturingDeposit(notional=-180_000_000, rate=0.005, max_maturity=10.0, decay_rate=0.20)
dynamic_deposits = NonMaturingDeposit(notional=-180_000_000, rate=0.005, max_maturity=10.0, decay_rate=0.20, beta_model=nmd_model)

# The 4 Bank Engines
engine_A = ALMEngine(portfolio=[naive_mortgages, static_deposits], baseline_curve=YieldCurve(tenors, base_rates))
engine_B = ALMEngine(portfolio=[ml_mortgages, static_deposits], baseline_curve=YieldCurve(tenors, base_rates))
engine_C = ALMEngine(portfolio=[naive_mortgages, dynamic_deposits], baseline_curve=YieldCurve(tenors, base_rates))
engine_D = ALMEngine(portfolio=[ml_mortgages, dynamic_deposits], baseline_curve=YieldCurve(tenors, base_rates))

# 3. Execute Rate Shocks
shocks = np.arange(-0.03, 0.035, 0.005)
results = []

for shock in shocks:
    shocked_rates = [max(0.0, r + shock) for r in base_rates]
    shocked_curve = YieldCurve(tenors, shocked_rates)

    results.append({
        'Shock (bps)': int(round(shock * 10000)),
        'Bank A (All Static)': engine_A.calculate_eve(shocked_curve),
        'Bank B (Dynamic Assets)': engine_B.calculate_eve(shocked_curve),
        'Bank C (Dynamic Liabilities)': engine_C.calculate_eve(shocked_curve),
        'Bank D (All Dynamic)': engine_D.calculate_eve(shocked_curve)
    })

df_results = pd.DataFrame(results).set_index('Shock (bps)')

# Normalize to Delta EVE
baseline_A = df_results.loc[0, 'Bank A (All Static)']
baseline_B = df_results.loc[0, 'Bank B (Dynamic Assets)']
baseline_C = df_results.loc[0, 'Bank C (Dynamic Liabilities)']
baseline_D = df_results.loc[0, 'Bank D (All Dynamic)']

df_results['Bank A (All Static)'] -= baseline_A
df_results['Bank B (Dynamic Assets)'] -= baseline_B
df_results['Bank C (Dynamic Liabilities)'] -= baseline_C
df_results['Bank D (All Dynamic)'] -= baseline_D

# 4. Plot the EVE profiles
plt.figure(figsize=(12, 7))

# Plot the 4 profiles
plt.plot(df_results.index, df_results['Bank A (All Static)'] / 1e6, label='Bank A: Static Prepayments and Static Deposit Betas', linestyle=':', color='gray', linewidth=2)
plt.plot(df_results.index, df_results['Bank B (Dynamic Assets)'] / 1e6, label='Bank B: ML Prepayments Only (Negative Convexity)', linestyle='--', color='dodgerblue', linewidth=2)
plt.plot(df_results.index, df_results['Bank C (Dynamic Liabilities)'] / 1e6, label='Bank C: Asymmetric NMD Betas (Margin Squeeze)', linestyle='--', color='darkorange', linewidth=2)
plt.plot(df_results.index, df_results['Bank D (All Dynamic)'] / 1e6, label='Bank D: Both Models Active', color='darkred', linewidth=3)

# Formatting
plt.axvline(0, color='black', linestyle='-', alpha=0.3)
plt.title('Economic Value of Equity (EVE) Stress Test\nIsolating Asset Risk vs Liability Risk', fontsize=14, fontweight='bold')
plt.xlabel('Parallel Rate Shock (bps)', fontsize=12)
plt.ylabel('Change in EVE ($ Millions)', fontsize=12)
plt.legend(fontsize=10, loc='best')
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()